# 03. Collect Teacher Trajectories On τ³-Bench Retail Train Tasks

This notebook creates the data that notebook 04 trains on.

The teacher is still served by vLLM. The benchmark environment and user simulator run locally through the τ³-Bench retail runtime. For every train task, we run the teacher inside the same harness we used for eval. Then we keep only successful trajectories and extract one supervised row per teacher assistant turn.

That row is not a raw string anymore. For MLX-LM, the clean format is:

```python
{
    "messages": [system, previous turns..., teacher assistant action],
    "tools": retail_tool_schemas,
    "task_id": "..."
}
```

MLX-LM's chat dataset can apply Qwen's chat template once and mask the prompt so only the final teacher action contributes loss.


In [1]:
from __future__ import annotations

import json

from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import retail_eval
from common import sft_rows
from common import tau_runtime
from common import teacher_backends
from common import user_simulator

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

TAU_BENCH_REPO_REVISION = cfg.TAU_BENCH_REPO_REVISION
TEACHER_MODEL = cfg.TEACHER_MODEL

print("Project root:", ROOT)
print("Blog dir:", BLOG_DIR)
print("Output dir:", OUTPUT_DIR)
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)
print("Pinned tau2-bench revision:", TAU_BENCH_REPO_REVISION[:8])

Project root: /Users/kargarisaac/codes/personal/distillation-blogs
Blog dir: /Users/kargarisaac/codes/personal/distillation-blogs/1-distilling-a-0-8b-tool-calling-agent
Output dir: /Users/kargarisaac/codes/personal/distillation-blogs/outputs
Loaded .env: True from /Users/kargarisaac/codes/personal/distillation-blogs/.env
Pinned tau2-bench revision: c42db6cc


## Runtime Backends

Start the teacher server in a terminal before running the collection cell:

```bash
source /Users/kargarisaac/.venv-vllm-metal/bin/activate
HF_HUB_DISABLE_XET=1 VLLM_METAL_MEMORY_FRACTION=0.95 vllm serve mlx-community/Qwen3.5-35B-A3B-8bit \
  --host 127.0.0.1 \
  --port 8092 \
  --max-model-len 81920 \
  --dtype float16 \
  --trust-remote-code \
  --enforce-eager \
  --generation-config vllm
```

The user simulator is configured in `.env` and served through the local ChatGPT subscription shim.


In [2]:
user_simulator_runtime = user_simulator.start_tau_bench_user_simulator_from_env(
    existing_shim=globals().get("chatgpt_user_simulator_shim")
)
chatgpt_user_simulator_shim = user_simulator_runtime.shim
TAU_BENCH_USER_SIMULATOR_LLM = user_simulator_runtime.model
TAU_BENCH_USER_SIMULATOR_ARGS = user_simulator_runtime.args
teacher_config = cfg.teacher_config_from_env(default_provider="vllm_raw", default_model=TEACHER_MODEL)

print("User simulator:", TAU_BENCH_USER_SIMULATOR_LLM)
print("User simulator args:", user_simulator.public_user_simulator_args(TAU_BENCH_USER_SIMULATOR_ARGS))
print("Teacher server:", teacher_config.server_base_url)
print("Teacher model:", teacher_config.model_name)
print("Teacher request model:", teacher_config.request_model)
print("Teacher temperature:", teacher_config.temperature)
print("Teacher top_p:", teacher_config.top_p)
print("Teacher top_k:", teacher_config.top_k)
print("Teacher max_new_tokens:", teacher_config.max_new_tokens)


User simulator: openai/gpt-5.4-mini
User simulator args: {'api_base': 'http://127.0.0.1:53033/v1', 'base_url': 'http://127.0.0.1:53033/v1', 'temperature': 0.0, 'num_retries': 6, 'timeout': 300}
Teacher server: http://127.0.0.1:8092
Teacher model: mlx-community/Qwen3.5-35B-A3B-8bit
Teacher request model: mlx-community/Qwen3.5-35B-A3B-8bit
Teacher temperature: 0.0
Teacher top_p: 1.0
Teacher top_k: 0
Teacher max_new_tokens: 2048


## Load Train Tasks And Retail Tools

The train split is the official τ³-Bench retail train split. The model never sees the retail database directly. It sees the policy, conversation history, tool schemas, and tool observations.


In [3]:
tokenizer = cfg.load_tokenizer()
retail_domain = tau_runtime.load_tau_bench_retail_domain(DATA_DIR, TAU_BENCH_REPO_REVISION)
tau_bench_runtime = retail_domain.runtime
retail_train_tasks = tau_bench_runtime.get_tasks("train")
retail_environment = retail_domain.environment
retail_policy = retail_domain.policy
retail_tools = retail_domain.tools
retail_tool_schema_by_name = retail_domain.tool_schema_by_name

print("Train tasks:", len(retail_train_tasks))
print("Retail tools:", len(retail_tools))
print("First train task id:", retail_train_tasks[0].id)
print("First five tools:", [tool["name"] for tool in retail_tools[:5]])


Train tasks: 74
Retail tools: 16
First train task id: 0
First five tools: ['calculate', 'cancel_pending_order', 'exchange_delivered_order_items', 'find_user_id_by_name_zip', 'find_user_id_by_email']


## Run Teacher On Train Tasks

This is intentionally cached task-by-task. If the run stops halfway, rerun the cell and it will continue from the saved JSON file.


In [ ]:
if not teacher_backends.teacher_runtime_is_configured(teacher_config):
    raise RuntimeError("Teacher server is not ready. Start the vLLM command above, then rerun this cell.")

TAU_BENCH_TEACHER_TRAIN_LIMIT = len(retail_train_tasks)
TAU_BENCH_TEACHER_TRAIN_MAX_STEPS = 100
TAU_BENCH_TEACHER_TRAIN_MAX_ERRORS = 10
TAU_BENCH_TEACHER_TRAIN_SEED = 42

user_simulator_slug = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
teacher_slug = cfg.filename_slug(teacher_config.model_name)
teacher_train_output_path = OUTPUT_DIR / f"{teacher_slug}_{teacher_config.provider}_tau3_bench_retail_train_teacher_trajectories_{user_simulator_slug}.json"
teacher_train_trace_dir = OUTPUT_DIR / "local_traces" / f"{teacher_slug}_{teacher_config.provider}_tau3_bench_retail_train_teacher_trajectories_{user_simulator_slug}"

teacher_train_config = cfg.TauBenchRetailEvalConfig(
    dataset_revision=TAU_BENCH_REPO_REVISION,
    student_model_name=teacher_config.model_name,
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
    max_steps=TAU_BENCH_TEACHER_TRAIN_MAX_STEPS,
    max_errors=TAU_BENCH_TEACHER_TRAIN_MAX_ERRORS,
    max_new_tokens=teacher_config.max_new_tokens,
    seed=TAU_BENCH_TEACHER_TRAIN_SEED,
    model_role="teacher_train",
)

teacher_train_task_objects = retail_train_tasks[:TAU_BENCH_TEACHER_TRAIN_LIMIT]
teacher_train_runner = retail_eval.TauBenchRetailTeacherEvalRunner(
    runtime=tau_bench_runtime,
    tokenizer=tokenizer,
    qwen_tools=retail_tools,
    tool_schema_by_name=retail_tool_schema_by_name,
    teacher_config=teacher_config,
    config=teacher_train_config,
    trace_dir=teacher_train_trace_dir,
)

print("Teacher train tasks:", len(teacher_train_task_objects))
print("Max steps per task:", TAU_BENCH_TEACHER_TRAIN_MAX_STEPS)
print("Max tool errors per task:", TAU_BENCH_TEACHER_TRAIN_MAX_ERRORS)
print("Seed:", TAU_BENCH_TEACHER_TRAIN_SEED)
print("Output path:", teacher_train_output_path)
print("Trace dir:", teacher_train_trace_dir)

teacher_train_payload = retail_eval.run_tau_bench_retail_eval_tasks(
    task_objects=teacher_train_task_objects,
    runner=teacher_train_runner,
    output_path=teacher_train_output_path,
    print_progress=True,
)

print("Teacher train successes:", teacher_train_payload["correct_count"], "/", teacher_train_payload["completed_count"])
print("Teacher train accuracy on train tasks:", round(teacher_train_payload["accuracy"], 3))

## Extract MLX-LM SFT Rows

Each successful trajectory becomes many training rows. If the teacher produced 12 assistant messages while solving one task, that one task can contribute 12 next-action imitation examples.


In [5]:
teacher_train_payload = json.loads(teacher_train_output_path.read_text(encoding="utf-8"))
teacher_sft_rows = sft_rows.extract_tau_bench_retail_sft_rows_from_eval_payload(
    teacher_train_payload,
    domain_policy=retail_policy,
    qwen_tools=retail_tools,
    only_successful_tasks=True,
    include_natural_language_targets=True,
    include_tool_call_targets=True,
)

teacher_sft_rows_path = OUTPUT_DIR / f"{teacher_slug}_{teacher_config.provider}_tau3_bench_retail_train_successful_sft_chat_rows_{user_simulator_slug}.jsonl"
cfg.write_jsonl(teacher_sft_rows_path, teacher_sft_rows)

success_task_ids = {row["task_id"] for row in teacher_train_payload.get("task_results", []) if row.get("is_success")}
tool_call_rows = sum(row["is_tool_call_target"] for row in teacher_sft_rows)
natural_language_rows = len(teacher_sft_rows) - tool_call_rows

print("Successful train tasks:", len(success_task_ids))
print("SFT rows:", len(teacher_sft_rows))
print("Tool-call target rows:", tool_call_rows)
print("Natural-language target rows:", natural_language_rows)
print("Saved SFT rows to:", teacher_sft_rows_path)

if teacher_sft_rows:
    print("\nFirst row roles:", [message["role"] for message in teacher_sft_rows[0]["messages"][-5:]])
    print("First target preview:\n", teacher_sft_rows[0]["target_text"][:800])


Successful train tasks: 31
SFT rows: 363
Tool-call target rows: 193
Natural-language target rows: 170
Saved SFT rows to: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/mlx_community_qwen3_5_35b_a3b_8bit_vllm_raw_tau3_bench_retail_train_successful_sft_chat_rows_openai_gpt_5_4_mini.jsonl

First row roles: ['system', 'assistant', 'user', 'assistant']
First target preview:
 <tool_call>
<function=get_order_details>
<parameter=order_id>
#W2378156
</parameter>
</function>
</tool_call>


## Token Lengths

These lengths matter for MLX training memory. The first tested Mac-native config in notebook 04 is conservative: bf16 base model, one LoRA layer, batch size 1, and prompt masking.


In [6]:
if teacher_sft_rows:
    lengths = [sft_rows.mlx_chat_row_token_length(row, tokenizer) for row in teacher_sft_rows]
    length_stats = {
        "min": min(lengths),
        "p50": cfg.percentile_int(lengths, 0.50),
        "p90": cfg.percentile_int(lengths, 0.90),
        "p95": cfg.percentile_int(lengths, 0.95),
        "max": max(lengths),
    }
    print(json.dumps(length_stats, indent=2))
else:
    print("No SFT rows yet. The teacher needs at least one successful train trajectory.")


{
  "min": 4938,
  "p50": 6184,
  "p90": 10094,
  "p95": 11081,
  "max": 15331
}
